In [86]:
"""
process_image — PyTorch‑only, auto‑threshold, 1‑px edges, full plots
====================================================================
* Depth & normal gradients via **torch conv2d Sobel**.
* Robust threshold (median + k·MAD) and sub‑pixel **non‑max suppression** — all GPU.
* Morphology (dilate/erode/close) with torch max_pool / min_pool.
* Region adjacency & plane similarity on GPU; CPU only for RANSAC and final Matplotlib.
* Produces the full eight‑panel figure (depth, normals, thin edges, CC, etc.) like your original.

Dependencies: PyTorch ≥ 2.1 (CUDA), Matplotlib; OpenCV only for connectedComponents and dilate in line‑coplanarity stage.
"""

from __future__ import annotations
import os, time, torch, matplotlib.pyplot as plt, networkx as nx, numpy as np, cv2
from typing import Tuple, List, Dict

from ground_truth.utility_methods import (
    reproject_depth_to_points, compute_normal_map_from_points,
    ransac_plane_fit, compute_distance_to_plane,
    find_line_planes, get_line_pixels_trim,
)
from ground_truth.dataloader import MogeGtLoader
from ground_truth.visualization import plot_lines_bool, plot_coplanar_lines

# ─── constants ─────────────────────────────────────────────────────────────
ANGLE_COS = np.cos(np.deg2rad(6))  # 6° similarity
DIST_EPS  = 0.02                   # 2 cm offset
MAD_K     = 3.5                    # robust factor
MIN_PTS   = 50

# ─── torch Sobel kernels ──────────────────────────────────────────────────
_SX = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)
_SY = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).view(1,1,3,3)

# ─── morphology helpers on GPU ─────────────────────────────────────────────

def _dilate(x: torch.Tensor, k: int = 3, it: int = 1):
    for _ in range(it):
        x = torch.nn.functional.max_pool2d(x, k, stride=1, padding=k//2)
    return x

def _erode(x: torch.Tensor, k: int = 3, it: int = 1):
    for _ in range(it):
        x = -torch.nn.functional.max_pool2d(-x, k, stride=1, padding=k//2)
    return x

# ─── robust auto‑threshold (GPU) ──────────────────────────────────────────

def _auto_thresh(mag: torch.Tensor, k: float = MAD_K):
    med = torch.median(mag)
    mad = torch.median(torch.abs(mag - med))
    mad = mad if mad > 1e-6 else torch.mean(torch.abs(mag - med))
    return med + k * mad

# ─── simple NMS for 1‑px edges (4 principal dirs) ─────────────────────────

def _nms(gx: torch.Tensor, gy: torch.Tensor, mag: torch.Tensor):
    ang = (torch.atan2(gy, gx) * 180/np.pi + 180) % 180  # 0‑180°
    keep = torch.zeros_like(mag, dtype=torch.bool)
    dirs = [(0,(0,1)), (45,(-1,1)), (90,(-1,0)), (135,(-1,-1))]
    for a0,(dy,dx) in dirs:
        m = (ang>=a0) & (ang<a0+45)
        fwd = torch.roll(mag, shifts=(dy,dx), dims=(2,3))
        back= torch.roll(mag, shifts=(-dy,-dx), dims=(2,3))
        keep |= m & (mag>=fwd) & (mag>=back)
    return keep

# ─── main ─────────────────────────────────────────────────────────────────

def process_image(image_dir:str, image_id:str, frame_str:str, net, device, *, dataset="hypersim", moge_model=None, half:bool=True):
    torch.cuda.empty_cache(); t0=time.perf_counter()

    # 1 ▸ load RGB & MoGe depth ------------------------------------------------
    if dataset=="hypersim": image_dir=os.path.join("data",image_id)
    color_np = MogeGtLoader(image_dir).load_color_image()      # H×W×3 uint8
    color_t  = torch.from_numpy(color_np).to(device).permute(2,0,1)/255
    color_t  = torch.nn.functional.interpolate(color_t[None], scale_factor=0.5, mode='bilinear', align_corners=False)[0]

    with torch.inference_mode(), torch.autocast('cuda', enabled=half):
        out = moge_model.infer(color_t)
    d = out['depth']                         # H×W
    depth = d.unsqueeze(0).unsqueeze(0)      # 1×1×H×W
    K = out['intrinsics'].cpu().numpy()
    print(f"MoGe: {time.perf_counter()-t0:.3f}s")

    # 2 ▸ normals -------------------------------------------------------------
    pts = reproject_depth_to_points(d.cpu().numpy(), K)
    normals_np = compute_normal_map_from_points(pts, ksize=1)
    normals_t  = torch.from_numpy(normals_np).to(device)

    # 3 ▸ Sobel gradients & NMS ----------------------------------------------
    gx_d = torch.nn.functional.conv2d(depth, _SX.to(device), padding=1)
    gy_d = torch.nn.functional.conv2d(depth, _SY.to(device), padding=1)
    mag_d= (gx_d**2+gy_d**2).sqrt()

    gray_n = (0.299*normals_t[...,2]+0.587*normals_t[...,1]+0.114*normals_t[...,0])[None,None]
    gx_n = torch.nn.functional.conv2d(gray_n, _SX.to(device), padding=1)
    gy_n = torch.nn.functional.conv2d(gray_n, _SY.to(device), padding=1)
    mag_n= (gx_n**2+gy_n**2).sqrt()

    th_d=_auto_thresh(mag_d).item(); th_n=_auto_thresh(mag_n).item()
    mask_d=_nms(gx_d,gy_d,mag_d)&(mag_d>th_d)
    mask_n=_nms(gx_n,gy_n,mag_n)&(mag_n>th_n)
    edges=(mask_d|mask_n).float()
    cc_mask=1-_erode(_dilate(edges,3,1),3,1)
    print(f"Grad+NMS: {time.perf_counter()-t0:.3f}s total so far")

    # 4 ▸ connected components via OpenCV ------------------------------------
    lbl_map = cv2.connectedComponents(cc_mask[0,0].cpu().numpy().astype(np.uint8))[1]

    # 5 ▸ batch‑GPU RANSAC per component -------------------------------------
    def _gpu_ransac(P: torch.Tensor, it: int = 48, eps: float = 0.03):
        """P: N×3 torch (CUDA) – returns best (n,d) or None"""
        N = P.shape[0]
        if N < MIN_PTS: return None
        idx = torch.randint(0, N, (it, 3), device=P.device)
        A = P[idx[:,0]]; B = P[idx[:,1]]; C = P[idx[:,2]]
        n = torch.nn.functional.normalize(torch.cross(B-A, C-A), dim=1)
        d = -(n * A).sum(dim=1)
        # guard degenerate triplets
        good = torch.isfinite(n).all(dim=1)
        n, d = n[good], d[good]
        if n.shape[0]==0: return None
        dist = torch.matmul(n, P.t()) + d.unsqueeze(1)  # (hyp × N)
        inliers = (dist.abs() < eps).float()
        best = inliers.sum(dim=1).argmax()
        if inliers[best].mean() < 0.8: return None
        n_best, d_best = n[best].cpu().numpy(), d[best].item()
        return n_best, d_best

    t_ransac = time.perf_counter()
    cluster_planes = {}
    pts_t = torch.from_numpy(pts).to(device)
    for lbl in range(1, lbl_map.max()+1):
        idx = torch.from_numpy((lbl_map == lbl)).to(device)
        if idx.sum() < MIN_PTS:
            continue
        model = _gpu_ransac(pts_t[idx])
        if model is None:
            continue
        n_best, d_best = model
        cluster_planes[lbl] = {"ls_model": (n_best, d_best)}
    print(f"GPU RANSAC: {time.perf_counter()-t_ransac:.3f}s  (labels={len(cluster_planes)})")

    # 6 ▸ GPU adjacency & merge ---------------------------------------------- & merge ----------------------------------------------
    valid=list(cluster_planes)
    groups=[]
    if valid:
        lm=torch.from_numpy(lbl_map).to(device)
        r=torch.stack((lm[:,:-1],lm[:,1:]),-1).view(-1,2)
        dwn=torch.stack((lm[:-1],lm[1:]),-1).view(-1,2)
        pairs=torch.cat((r,dwn),0); pairs=pairs[pairs[:,0]!=pairs[:,1]]
        mask=torch.isin(pairs,torch.tensor(valid,device=device))
        pairs=pairs[mask.all(1)]
        pairs=torch.unique(torch.where(pairs[:,0]<pairs[:,1], pairs[:,0]*100000+pairs[:,1], pairs[:,1]*100000+pairs[:,0]))
        a=(pairs//100000).cpu().numpy(); b=(pairs%100000).cpu().numpy(); adj=set(zip(a,b))
        lbls=torch.tensor(valid,device=device)
        norms=torch.stack([torch.tensor(cluster_planes[int(l)]["ls_model"][0],device=device) for l in lbls])
        offs =torch.tensor([cluster_planes[int(l)]["ls_model"][1] for l in lbls],device=device)
        sim=((norms@norms.T).abs()>ANGLE_COS)&((offs[:,None]-offs).abs()<DIST_EPS)
        G=nx.Graph(); G.add_nodes_from(valid)
        for a,b in adj:
            ia=(lbls==a).nonzero(as_tuple=False).item(); ib=(lbls==b).nonzero(as_tuple=False).item()
            if sim[ia,ib]:G.add_edge(a,b)
        groups=list(nx.connected_components(G))

    merged=np.zeros_like(lbl_map)
    for new,grp in enumerate(groups,1):
        for g in grp: merged[lbl_map==g]=new
    dil=cv2.dilate((merged>0).astype(np.uint8),np.ones((3,3),np.uint8),iterations=3)*merged

    # 7 ▸ DeepLSD lines & coplanar labels ------------------------------------
    gray=(0.299*color_t[0]+0.587*color_t[1]+0.114*color_t[2]).cpu().numpy()
    inp=torch.tensor(gray,device=device)[None,None]
    with torch.no_grad():
        out_lines = net({'image': inp})
    lines = out_lines['lines'][0]
    if isinstance(lines, torch.Tensor):
        lines = lines.cpu().numpy()
    line_labels = find_line_planes([l.reshape(2, 2) if l.shape == (4,) else l for l in lines], dil, get_line_pixels_trim)

    # 8 ▸ full plots ----------------------------------------------------------
    plt.figure(figsize=(20,25))
    plt.subplot(4,2,1);plt.title('Depth');plt.imshow(d.cpu(),cmap='gray');plt.axis('off')
    plt.subplot(4,2,2);plt.title('Normals');plt.imshow((normals_t.cpu()+1)/2,cmap='gray');plt.axis('off')
    plt.subplot(4,2,3);plt.title('Edges depth');plt.imshow(mask_d[0,0].cpu(),cmap='gray');plt.axis('off')
    plt.subplot(4,2,4);plt.title('Edges normal');plt.imshow(mask_n[0,0].cpu(),cmap='gray');plt.axis('off')
    plt.subplot(4,2,5);plt.title('Combined');plt.imshow(edges[0,0].cpu(),cmap='gray');plt.axis('off')

    ax=plt.subplot(4,2,8);plt.title('Lines');plt.axis('off')
    plot_lines_bool(ax,color_np,lines,is_correct=[True]*len(lines))
    plt.show()

    # CC plots ---------------------------------------------------------------
    np.random.seed(22); lc=np.random.randint(0,255,(merged.max()+1,3),dtype=np.uint8)
    plt.figure(figsize=(20,25))
    plt.subplot(4,3,2);plt.title('CC');plt.imshow(cv2.cvtColor(lc[lbl_map],cv2.COLOR_BGR2RGB));plt.axis('off')
    plt.subplot(4,3,4);plt.title('Planar CC');plt.imshow(cv2.cvtColor(lc[merged],cv2.COLOR_BGR2RGB));plt.axis('off')
    plt.subplot(4,3,5);plt.title('Dilated');plt.imshow(cv2.cvtColor(lc[dil],cv2.COLOR_BGR2RGB));plt.axis('off')
    ax=plt.subplot(4,3,6);plt.title('Coplanar');plt.axis('off')
    plot_coplanar_lines(ax,[l.reshape(2,2) if l.shape==(4,) else l for l in lines],line_labels,color_np)
    plt.subplot(4,3,8);plt.title('Original');plt.imshow(cv2.cvtColor(color_np,cv2.COLOR_BGR2RGB));plt.axis('off')
    plt.show()

    return merged, cluster_planes, groups, line_labels


In [87]:
import numpy as np
import cv2
import hdbscan
from sklearn.neighbors import NearestNeighbors
from sklearn.impute import SimpleImputer
import numpy as np
import matplotlib.pyplot as plt
import hdbscan
import matplotlib
import random
import os
import json
import torch 
import h5py
import glob
from numpy import linalg as LA

random.seed(10)

# Exploration

In [88]:
import os
import torch
from deeplsd.models.deeplsd_inference import DeepLSD


In [89]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
conf = {'detect_lines': True, 'line_detection_params': {'merge': False, 'filtering': True, 'grad_thresh': 3}}
ckpt = torch.load('../weights/deeplsd_md.tar', map_location='cpu', weights_only=False)
net = DeepLSD(conf)
net.load_state_dict(ckpt['model'])
net = net.to(device).eval()


In [ ]:
frame_str = "0001"
desired_images = [
    "ai_001_001",
    "ai_001_004",
    "ai_001_005",
    "ai_001_006",
    "ai_001_007",
    "ai_001_008",
    "ai_001_009",
    "ai_002_001",

    #"DSC_0442",
    #"DSC_0455",
    #"DSC_0239",
    #"DSC_0249",
]

desired_images = [
    #"ai_001_001",
    #"ai_001_004",
    #"ai_001_005",
    #"ai_001_006",
    #"ai_001_007",
    #"ai_001_008",
    #"ai_001_009",
    #"ai_002_001",

   
    "DSC_0239",
    "DSC_0249",
    "DSC_0298",
    "DSC_0299",
    "DSC_0300",
    "DSC_0340",
    "DSC_0341",
    "DSC_0342",
    "DSC_0442",
    "DSC_0455",
]
from moge.model.v1 import MoGeModel

moge = MoGeModel.from_pretrained("Ruicheng/moge-vitl").to(device)

# Process, plot, and save JSON data for each image.
for image_id in desired_images:
    #composite_after, pred_lines, img, normals, world_coordinates, valid_mask, line_info, scores, isstruct, original_lines
    # Load the model from huggingface hub (or load from local).
    image_dir = os.path.join("data", image_id)
    merged, cluster_planes, groups, line_labels, cluster_planes, groups, line_labels = process_image(
        image_dir, image_id, frame_str, net, device,
        dataset="hypersim",
        moge_model = moge
    )
    

[04/23/2025 20:40:03 INFO] using MLP layer as FFN
